# SolarSight - GPU training on Google Colab (free T4)

The repo's `models/best.pt` came from a **reduced CPU run** (10 epochs @ 320px - see docs/TRAINING.md).
This notebook trains the **full-quality model**: YOLO11-nano, 640px, 60 epochs, on a free Colab GPU (~25 min).

**Steps:** Runtime > Change runtime type > **T4 GPU** > Run all. Download `best.pt` at the end and drop it into `models/`.

In [ ]:
# 1. Confirm we actually have a GPU (never train on CPU by accident)
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Install ultralytics (Colab already has torch+cuda)
%pip install -q ultralytics kagglehub

In [ ]:
# 3. Download the dataset anonymously (no Kaggle account needed)
#    Thermal Solar PV Anomaly Detection Dataset, CC BY-NC 4.0
import kagglehub, shutil
from pathlib import Path

cache = Path(kagglehub.dataset_download("pkdarabi/solarpanel"))
src = cache / "ImageSet" if (cache / "ImageSet").is_dir() else cache
dst = Path("/content/dataset")
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print("Dataset at", dst)
for split in ["train", "valid", "test"]:
    n = len(list((dst / split / "images").glob("*.jpg")))
    print(f"{split}: {n} images")

In [ ]:
# 4. Train: YOLO11-nano fine-tune, 640px, 60 epochs (~25 min on T4)
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # COCO-pretrained nano
model.train(
    data="/content/dataset/data.yaml",
    epochs=60,
    imgsz=640,            # native dataset size: full detail for small hotspots
    batch=32,             # T4 has 16 GB; lower to 16 if OOM
    device=0,
    seed=42,
    patience=20,          # early stop if no improvement
    project="/content/runs",
    name="solarsight",
)

In [ ]:
# 5. Evaluate on the held-out TEST split and print per-class metrics
best = "/content/runs/solarsight/weights/best.pt"
model = YOLO(best)
metrics = model.val(data="/content/dataset/data.yaml", split="test", plots=True)
print("mAP50:", round(float(metrics.box.map50), 3))
print("mAP50-95:", round(float(metrics.box.map), 3))
for i, name in metrics.names.items():
    print(f"{name:24s} P={metrics.box.p[i]:.3f} R={metrics.box.r[i]:.3f} "
          f"mAP50={metrics.box.ap50[i]:.3f} mAP50-95={metrics.box.ap[i]:.3f}")

In [ ]:
# 6. Download the weights for the repo (drop into models/best.pt)
from google.colab import files
files.download(best)